# Bullish Pattern Reranking Backtest

Runs the causal bullish-pattern layer after the existing Kronos candidate screener for both archived 1D models. The weight search uses the complete timeframe, matching the existing backtest objective.

In [1]:
from pathlib import Path
import subprocess
import sys

cwd = Path.cwd().resolve()
repo = cwd.parent if cwd.name == 'BackTest Pattern Screener' else cwd
pattern_dir = repo / 'BackTest Pattern Screener'
assert pattern_dir.exists(), f'Pattern folder not found from {cwd}'
print('Repository:', repo)

Repository: /Users/mraffyzeidan/Downloads/ISTL


In [2]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(pattern_dir / 'requirements.txt')],
    check=True,
)

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


CompletedProcess(args=['/Library/Developer/CommandLineTools/usr/bin/python3', '-m', 'pip', 'install', '-q', '-r', '/Users/mraffyzeidan/Downloads/ISTL/BackTest Pattern Screener/requirements.txt'], returncode=0)

In [3]:
subprocess.run(
    [sys.executable, str(pattern_dir / 'run_pattern_backtest.py'), '--repo', str(repo)],
    check=True,
)

Calculating one causal pattern snapshot for all models...
validated_no_refit_e15_1d           ranking  top_k  selections  hits  precision  daily_basket_success                  run_name      lift
             base      1          41    14   0.341463              0.341463 validated_no_refit_e15_1d  0.000000
             base      3         123    57   0.463415              0.780488 validated_no_refit_e15_1d  0.000000
             base      5         205   101   0.492683              0.926829 validated_no_refit_e15_1d  0.000000
             base     10         410   195   0.475610              0.975610 validated_no_refit_e15_1d  0.000000
             base     20         820   387   0.471951              0.975610 validated_no_refit_e15_1d  0.000000
             base     30        1230   584   0.474797              0.975610 validated_no_refit_e15_1d  0.000000
base_plus_pattern      1          41    22   0.536585              0.536585 validated_no_refit_e15_1d  0.195122
base_plus_pattern   

CompletedProcess(args=['/Library/Developer/CommandLineTools/usr/bin/python3', '/Users/mraffyzeidan/Downloads/ISTL/BackTest Pattern Screener/run_pattern_backtest.py', '--repo', '/Users/mraffyzeidan/Downloads/ISTL'], returncode=0)

In [4]:
import pandas as pd

comparison = pd.read_csv(pattern_dir / 'results' / 'all_model_comparison.csv')
comparison.pivot_table(
    index=['run_name', 'top_k'],
    columns='ranking',
    values=['precision', 'daily_basket_success'],
).round(4)

daily_basket_success                    \
ranking                                         base base_plus_pattern   
run_name                  top_k                                          
production_refit_e4_1d    1                   0.4390            0.5610   
                          3                   0.8537            0.9512   
                          5                   0.9268            0.9512   
                          10                  0.9756            0.9756   
                          20                  0.9756            0.9756   
                          30                  0.9756            0.9756   
validated_no_refit_e15_1d 1                   0.3415            0.5366   
                          3                   0.7805            0.8780   
                          5                   0.9268            0.9756   
                          10                  0.9756            0.9756   
                          20                  0.9756            0.9756   
                          30                  0.9756            0.9756   

                                precision                    
ranking                              base base_plus_pattern  
run_name                  top_k                              
production_refit_e4_1d    1        0.4390            0.5610  
                          3        0.5366            0.5610  
                          5        0.5268            0.5561  
                          10       0.4902            0.4878  
                          20       0.4732            0.4671  
                          30       0.4715            0.4577  
validated_no_refit_e15_1d 1        0.3415            0.5366  
                          3        0.4634            0.5203  
                          5        0.4927            0.5415  
                          10       0.4756            0.5390  
                          20       0.4720            0.4866  
                          30       0.4748            0.4642

In [5]:
latest_examples = []
for run_name in ('validated_no_refit_e15_1d', 'production_refit_e4_1d'):
    selected = pd.read_csv(pattern_dir / 'results' / run_name / 'selected_top30_pattern.csv')
    latest = selected['origin'].max()
    view = selected.loc[(selected['origin'] == latest) & (selected['selected_rank'] <= 10)].copy()
    view.insert(0, 'model', run_name)
    latest_examples.append(view)
pd.concat(latest_examples)[[
    'model', 'origin', 'selected_rank', 'ticker', 'secondary_score',
    'net_pattern_score', 'final_ranking_score', 'top_bullish_pattern',
    'pattern_support', 'pattern_penalty'
]]

,model,origin,selected_rank,ticker,secondary_score,net_pattern_score,final_ranking_score,top_bullish_pattern,pattern_support,pattern_penalty
1200,validated_no_refit_e15_1d,2026-07-29,1,TNCA,0.324055,0.207690,0.9490,bearish_structure_break,volume_expansion,low_liquidity
1201,validated_no_refit_e15_1d,2026-07-29,2,WMPP,0.333855,0.062115,0.9450,higher_low,volume_expansion,gap_up
1202,validated_no_refit_e15_1d,2026-07-29,3,BULL,0.264574,0.346415,0.8750,higher_low,bullish_structure,close_below_high
1203,validated_no_refit_e15_1d,2026-07-29,4,BAJA,0.267701,0.107258,0.8705,bullish_hh_hl_structure,volume_expansion,close_below_high
1204,validated_no_refit_e15_1d,2026-07-29,5,TFAS,0.266662,0.007958,0.8570,higher_low,volume_expansion,close_below_high
1205,validated_no_refit_e15_1d,2026-07-29,6,CASH,0.235305,0.212310,0.8030,higher_low,bullish_structure,low_liquidity
1206,validated_no_refit_e15_1d,2026-07-29,7,ELPI,0.347269,0.000000,0.8005,none,volume_expansion,close_below_high
1207,validated_no_refit_e15_1d,2026-07-29,8,NIRO,0.338937,0.000000,0.7940,none,volume_expansion,low_liquidity
1208,validated_no_refit_e15_1d,2026-07-29,9,HATM,0.218218,0.852335,0.7820,bullish_hh_hl_structure,bullish_structure,extended
1209,validated_no_refit_e15_1d,2026-07-29,10,EPAC,0.323574,0.000000,0.7745,none,none,close_below_high
